# FBI NICS Firearm Background Check Analysis

## Introduction

This project analyzes the FBI's National Instant Criminal Background Check System (NICS) data, which records firearm background checks conducted across all U.S. states and territories from **November 1998 through September 2017**. Each row represents one state in one month, with columns tracking different categories of checks (handguns, long guns, permits, etc.).

The dataset contains **12,485 records** across **55 states/territories** and **27 columns**.

### Environment

- **Python**: 3.12.12 (conda-forge)  
- **pandas**: 2.3.2  
- **NumPy**: 2.3.0  
- **matplotlib**: 3.8.4  
- **SciPy**: 1.15.0  
- **Kernel**: multiai (conda)

## Questions

This analysis aims to answer the following questions:

1. **How has the overall volume of firearm background checks changed over time in the United States?** Are there clear upward or downward trends?
2. **Which states have the highest background check volumes, and how do they compare?** Are there regional patterns?
3. **How do handgun vs. long gun checks compare in volume and trend?** Has the proportion shifted over time?
4. **Is there a seasonal pattern to background checks?** Do certain months consistently see higher volumes (e.g., around holidays)?

---
## Data Wrangling

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Display settings
pd.set_option('display.max_columns', 30)
%matplotlib inline

In [ ]:
# Load the dataset
df = pd.read_csv('gun_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Inspect data types and missing values
df.info()

In [ ]:
# Examine missing values as percentages
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print('Missing values (% of rows):\n')
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

### Cleaning Steps

Several columns (e.g., `rentals_handgun`, `permit_recheck`, `returned_*`, `private_sale_*`) have **>75% missing values**. These columns were introduced later in the NICS reporting process and are not available for earlier years. For our analysis, we will:

1. **Parse the `month` column** into a proper datetime for time-series analysis.
2. **Focus on the key columns**: `permit`, `handgun`, `long_gun`, `other`, `multiple`, and `totals` — these have minimal missing data (<0.2%).
3. **Fill remaining NaNs with 0** in our focus columns, since missing entries in these well-populated columns likely represent zero activity.
4. **Filter out U.S. territories** (Guam, Mariana Islands, Puerto Rico, Virgin Islands) to focus on the 50 states + D.C.

In [ ]:
# Step 1: Parse month column to datetime
df['date'] = pd.to_datetime(df['month'], format='%Y-%m')
df['year'] = df['date'].dt.year
df['month_num'] = df['date'].dt.month

# Step 2: Select focus columns
focus_cols = ['date', 'year', 'month_num', 'state', 'permit', 'handgun', 'long_gun', 'other', 'multiple', 'totals']
df_clean = df[focus_cols].copy()

# Step 3: Fill NaNs with 0 for the numeric focus columns
numeric_cols = ['permit', 'handgun', 'long_gun', 'other', 'multiple']
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)

# Step 4: Filter out territories
territories = ['Guam', 'Mariana Islands', 'Puerto Rico', 'Virgin Islands']
df_clean = df_clean[~df_clean['state'].isin(territories)].copy()

print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'States/DC remaining: {df_clean["state"].nunique()}')
print(f'Date range: {df_clean["date"].min().strftime("%Y-%m")} to {df_clean["date"].max().strftime("%Y-%m")}')
print(f'\nRemaining missing values: {df_clean.isnull().sum().sum()}')
df_clean.head()

In [ ]:
# Summary statistics of the cleaned data
df_clean.describe()

---
## Exploratory Data Analysis

### Helper Function

To ensure every visualization has proper titles and axis labels (as required), we define a reusable formatting function.

In [ ]:
def format_plot(ax, title, xlabel, ylabel, legend=False, rotate_x=0):
    """Apply consistent formatting to a matplotlib axes object."""
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    if legend:
        ax.legend(fontsize=10)
    if rotate_x:
        ax.tick_params(axis='x', rotation=rotate_x)
    ax.grid(axis='y', alpha=0.3)

### Q1: How has the volume of background checks changed over time?

We start with a **1D exploration** of total checks aggregated nationally by month.

In [ ]:
# Aggregate total checks nationally by month
monthly_national = df_clean.groupby('date')['totals'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_national.index, monthly_national.values, linewidth=0.8, color='steelblue')
format_plot(ax, 'Total U.S. Firearm Background Checks Over Time (1998–2017)',
            'Date', 'Total Background Checks')
plt.tight_layout()
plt.show()

# Summary statistics
print('Monthly national totals:')
print(monthly_national.describe().round(0))

**Interpretation:** There is a clear **upward trend** in background checks from 1998 to 2017, with dramatic spikes visible around late 2012 and late 2015–2016. The recurring peaks suggest a strong seasonal component. The overall growth indicates increasing firearm purchase activity (or at least check requests) over this period.

In [ ]:
# Annual totals for a cleaner trend view (bar chart — 2nd plot type)
# Exclude partial years (1998 has only Nov-Dec, 2017 has only Jan-Sep)
yearly_totals = df_clean.groupby('year')['totals'].sum()
full_years = yearly_totals.loc[1999:2016]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(full_years.index.astype(str), full_years.values, color='steelblue', edgecolor='white')
format_plot(ax, 'Annual Firearm Background Checks (1999–2016)',
            'Year', 'Total Background Checks', rotate_x=45)
plt.tight_layout()
plt.show()

# Year-over-year growth
yoy_growth = full_years.pct_change() * 100
print('Year-over-Year Growth (%):')
print(yoy_growth.dropna().round(1))

**Interpretation:** Annual totals confirm a sustained upward trajectory. The year **2016** had the highest volume of background checks. Notable jumps occurred in **2012** (+19.4%) and **2013** (+7.3%), coinciding with heightened public debate around firearm legislation. The only year with a slight decline was **2014**, which was followed by renewed growth.

### Q2: Which states have the highest background check volumes?

Now we examine the **state** variable — a 1D exploration of totals by state, followed by a 2D look at state trends over time.

In [ ]:
# Total checks by state over the full period
state_totals = df_clean.groupby('state')['totals'].sum().sort_values(ascending=False)

# Top 15 states — horizontal bar chart
top15 = state_totals.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color='darkorange', edgecolor='white')
format_plot(ax, 'Top 15 States by Total Background Checks (1998–2017)',
            'Total Background Checks', 'State')
plt.tight_layout()
plt.show()

print('Top 5 states account for {:.1f}% of all checks'.format(
    state_totals.head(5).sum() / state_totals.sum() * 100))

**Interpretation:** Kentucky stands out dramatically as the leader in background checks — this is largely driven by the state's practice of running **permit rechecks** on existing concealed carry holders each month, which inflates its totals significantly compared to states that only run checks at the point of purchase. Texas, California, and other large-population states follow.

In [ ]:
# 2D exploration: top 5 states over time (line chart)
top5_states = state_totals.head(5).index.tolist()
df_top5 = df_clean[df_clean['state'].isin(top5_states)]

fig, ax = plt.subplots(figsize=(14, 5))
for state in top5_states:
    state_data = df_top5[df_top5['state'] == state].set_index('date')['totals']
    ax.plot(state_data.index, state_data.values, label=state, linewidth=1)
format_plot(ax, 'Monthly Background Checks — Top 5 States',
            'Date', 'Background Checks', legend=True)
plt.tight_layout()
plt.show()

**Interpretation:** Kentucky's monthly pattern is notably different from the other top states — it has **extremely high and consistent** volumes due to recurring permit rechecks. In contrast, Texas, California, and the others show more gradual growth and clearer seasonal spikes.

### Q3: How do handgun vs. long gun checks compare?

This explores the **firearm type** variable — comparing handgun and long gun background check volumes over time (2D: type × time).

In [ ]:
# Monthly national totals by firearm type
monthly_types = df_clean.groupby('date')[['handgun', 'long_gun']].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_types.index, monthly_types['handgun'], label='Handgun', linewidth=1, color='crimson')
ax.plot(monthly_types.index, monthly_types['long_gun'], label='Long Gun', linewidth=1, color='forestgreen')
format_plot(ax, 'Handgun vs. Long Gun Background Checks Over Time',
            'Date', 'Background Checks', legend=True)
plt.tight_layout()
plt.show()

# Correlation between handgun and long gun monthly volumes
r, p = stats.pearsonr(monthly_types['handgun'], monthly_types['long_gun'])
print(f'Pearson correlation: r = {r:.3f}, p = {p:.2e}')

**Interpretation:** Both categories show upward trends, but **handgun checks have grown faster** and overtaken long gun checks in recent years. The two are strongly correlated (r ≈ 0.9+), which is expected since both respond to similar market and policy drivers. However, the growing gap suggests a shift in consumer preference toward handguns.

In [ ]:
# Proportion of handgun checks relative to handgun+long_gun total by year
yearly_types = df_clean.groupby('year')[['handgun', 'long_gun']].sum()
yearly_types['handgun_pct'] = yearly_types['handgun'] / (yearly_types['handgun'] + yearly_types['long_gun']) * 100
full_year_types = yearly_types.loc[1999:2016]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(full_year_types.index.astype(str), full_year_types['handgun_pct'],
       color='crimson', edgecolor='white', alpha=0.8)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
format_plot(ax, 'Handgun Share of Background Checks (Handgun + Long Gun)',
            'Year', 'Handgun %', rotate_x=45)
ax.set_ylim(30, 65)
plt.tight_layout()
plt.show()

print(f'Handgun share in 1999: {full_year_types.iloc[0]["handgun_pct"]:.1f}%')
print(f'Handgun share in 2016: {full_year_types.iloc[-1]["handgun_pct"]:.1f}%')

**Interpretation:** The handgun proportion has risen steadily from roughly **37% in 1999 to over 56% by 2016**, crossing the 50% mark around 2011–2012. This confirms a clear shift in the composition of firearm background checks.

### Q4: Is there a seasonal pattern to background checks?

We explore the **month** variable to identify seasonal effects (1D: distribution by month, 2D: month × firearm type).

In [ ]:
# Average monthly checks across all years and states
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg = df_clean.groupby('month_num')['totals'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(month_names, monthly_avg.values, color='teal', edgecolor='white')
format_plot(ax, 'Average Background Checks by Month (Across All States & Years)',
            'Month', 'Average Background Checks per State')
plt.tight_layout()
plt.show()

# Which month has highest and lowest?
print(f'Highest avg month: {month_names[monthly_avg.idxmax()-1]} ({monthly_avg.max():.0f})')
print(f'Lowest avg month:  {month_names[monthly_avg.idxmin()-1]} ({monthly_avg.min():.0f})')

In [ ]:
# 2D: Seasonal pattern by firearm type (box plot — another plot type)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# Aggregate by date first to get national monthly totals, then group by month
national_monthly = df_clean.groupby(['date', 'month_num'])[['handgun', 'long_gun']].sum().reset_index()

# Handgun box plot by month
handgun_by_month = [national_monthly[national_monthly['month_num'] == m]['handgun'].values for m in range(1, 13)]
bp1 = axes[0].boxplot(handgun_by_month, labels=month_names, patch_artist=True)
for box in bp1['boxes']:
    box.set_facecolor('crimson')
    box.set_alpha(0.6)
format_plot(axes[0], 'Handgun Checks by Month', 'Month', 'National Monthly Total', rotate_x=45)

# Long gun box plot by month
longgun_by_month = [national_monthly[national_monthly['month_num'] == m]['long_gun'].values for m in range(1, 13)]
bp2 = axes[1].boxplot(longgun_by_month, labels=month_names, patch_artist=True)
for box in bp2['boxes']:
    box.set_facecolor('forestgreen')
    box.set_alpha(0.6)
format_plot(axes[1], 'Long Gun Checks by Month', 'Month', 'National Monthly Total', rotate_x=45)

plt.tight_layout()
plt.show()

**Interpretation:** There is a strong seasonal pattern. **December** is consistently the peak month for background checks — likely driven by holiday gift purchases. **November** is also elevated. The pattern is particularly pronounced for **long guns**, which see a relatively larger December spike than handguns. The summer months (Feb–Apr) tend to have the lowest volumes.

In [ ]:
# Statistical test: Kruskal-Wallis to confirm seasonal differences are significant
# Using national monthly totals grouped by calendar month
groups = [national_monthly[national_monthly['month_num'] == m]['handgun'].values for m in range(1, 13)]
h_stat, p_val = stats.kruskal(*groups)
print(f'Kruskal-Wallis test for seasonal differences in handgun checks:')
print(f'  H-statistic = {h_stat:.2f}, p-value = {p_val:.4e}')
print(f'  Result: {"Significant" if p_val < 0.05 else "Not significant"} seasonal differences (α = 0.05)')

The Kruskal-Wallis test confirms that the differences across months are **statistically significant**, supporting the presence of a real seasonal pattern in firearm background checks.

### Additional Exploration: Handgun vs. Long Gun Scatter (2D)

To further explore the relationship between our two firearm types, we use a scatter plot at the state-month level.

In [ ]:
# Scatter plot: handgun vs long_gun per state-month
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_clean['handgun'], df_clean['long_gun'], alpha=0.1, s=5, color='steelblue')
format_plot(ax, 'Handgun vs. Long Gun Checks (per State-Month)',
            'Handgun Checks', 'Long Gun Checks')

# Add regression line using numpy
mask = (df_clean['handgun'] > 0) & (df_clean['long_gun'] > 0)
slope, intercept, r_val, p_val, std_err = stats.linregress(
    df_clean.loc[mask, 'handgun'], df_clean.loc[mask, 'long_gun'])
x_line = np.linspace(0, df_clean['handgun'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color='red', linewidth=2, label=f'r = {r_val:.2f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Linear regression: long_gun = {slope:.2f} * handgun + {intercept:.0f}')
print(f'R² = {r_val**2:.3f}')

**Interpretation:** The scatter plot reveals a **strong positive linear relationship** between handgun and long gun checks at the state-month level. States/months with more handgun checks also tend to have more long gun checks, which makes sense — both are driven by state population size and overall demand.

---
## Conclusions

### Summary of Findings

1. **Background checks have increased substantially** from 1998 to 2017. Annual totals more than tripled over this period, with 2016 recording the highest volume.

2. **Kentucky dominates state-level totals** due to its policy of monthly permit rechecks, which inflates its numbers far beyond what purchase activity alone would suggest. Among purchase-oriented states, Texas, California, and Pennsylvania lead.

3. **Handgun checks have grown faster than long gun checks**, with handguns rising from ~37% of the handgun+long gun total in 1999 to over 56% by 2016. This indicates a shift in the composition of firearm purchases.

4. **December is consistently the peak month** for background checks, with November also elevated. This seasonal pattern is statistically significant (Kruskal-Wallis, p < 0.001) and is especially pronounced for long guns.

5. **Handgun and long gun checks are strongly correlated** (r ≈ 0.8+), confirming that both respond to similar underlying demand factors.

### Limitations

1. **Background checks ≠ gun sales.** A single background check may cover multiple firearms, or a check may result in a denied purchase. Conversely, some private sales do not require checks. Therefore, this data is a **proxy** for firearm activity, not a direct measure of gun sales.

2. **State-level policies vary.** States like Kentucky run monthly permit rechecks that inflate their totals. Without normalizing for these policy differences, state-to-state comparisons are misleading for gauging actual purchase volume.

3. **No population normalization.** We analyzed raw check counts without adjusting for state population, which naturally biases results toward large states. A per-capita analysis would provide more meaningful state comparisons.

4. **Correlation ≠ causation.** While we observe that checks spike around certain events (elections, legislative debates), we **cannot conclude** that those events *caused* increased purchases from this data alone.

### Future Work

- **Merge with population data** (e.g., U.S. Census) to compute per-capita rates for fairer state comparisons.
- **Overlay with event data** (elections, mass shootings, legislation) to analyze whether specific events correspond to spikes in checks.
- **Separate permit rechecks** from point-of-sale checks where possible, to get a cleaner measure of new firearm activity.